# 08 · Wikipedia pageviews (task 5.8, first outside source)

Daily views of English Wikipedia articles by people, since July 2015, from the Wikimedia API (no key). Three uses:
- **how known a topic was before it trended**: the median daily views of its article over the 28 days before, a feature for lifespan (a famous name and a new one fade differently);
- **how big it got**: views on the day it trended against that baseline (Wikimedia publishes each day's figures about a day later, so the newest days show as unknown);
- **seasons and calendar moments**: which months an article peaks in across years (RQ5's seasons, the weeks horizon).

Topics are linked by Wikipedia's search. Use only `exact` links that aren't disambiguation pages: they're right for nearly every name and place. A lone first name ("Reggie") is still ambiguous. `partial` links are about half right, and long sentence-like titles from Bluesky mostly fail.

Requests go one at a time with a descriptive User-Agent (the repository's URL) and are cached in `research/data/wiki/`, so a re-run fetches nothing twice. Run top to bottom. Clear the outputs before committing (research/README.md).

In [ ]:
from pathlib import Path

import pandas as pd

from topnews import dataset, db, wiki

pd.set_option("display.max_colwidth", 45)
engine = db.engine()  # read-only; RESEARCH_DATABASE_URL from .env.local
snaps, topics = db.snapshots(engine), db.topics(engine)
if snaps.empty:  # before production has snapshots: the replay (scripts/replay-snapshots.ts)
    snaps, topics = dataset.load_replay(db.REPO_ROOT / "research" / "data" / "replay")
first_top10 = snaps[snaps.position <= 10].groupby("topic_id").taken_at.min()
linked = topics.set_index("id").loc[first_top10.index, ["label"]].assign(trended=first_top10)
get = wiki.cached()
found = [wiki.find_article(label, get) or {"title": None, "disambiguation": None, "quality": "not found"} for label in linked.label]
linked = linked.assign(**pd.DataFrame(found, index=linked.index))
linked["usable"] = linked.quality.eq("exact") & linked.disambiguation.eq(False)
print(f"{len(linked)} topics that reached the top 10; {int(linked.usable.sum())} with a usable link")
linked.groupby("quality").size()

In [ ]:
linked.sort_values(["quality", "label"])[["label", "title", "quality", "disambiguation"]]

## How known before, and how big on the day

In [ ]:
rows = []
for topic_id, row in linked[linked.usable].iterrows():
    day = row.trended.tz_convert(None).normalize()
    views = wiki.daily_views(row.title, day - pd.to_timedelta(28, unit="D"), day, get)
    rows.append({"label": row.label, "article": row.title, **wiki.spike(views, day)})
sizes = pd.DataFrame(rows).sort_values("baseline", ascending=False)
sizes.round(1)

## Seasons and calendar moments

Articles behind the calendar moments RQ4 found, over full years: the months they come back in. `vs_overall` is the month's median daily views against the article's overall median.

In [ ]:
CALENDAR = ["Autumn", "International Coffee Day", "National Day for Truth and Reconciliation", "Fat Bear Week"]
profiles = {}
for name in CALENDAR:
    title = wiki.find_article(name, get)["title"]  # the article itself, not a redirect
    views = wiki.daily_views(title, pd.Timestamp("2016-01-01"), pd.Timestamp("2025-12-31"), get)
    profiles[title] = wiki.monthly_profile(views).set_index("month")["vs_overall"]
pd.DataFrame(profiles).round(2)